# Hardware Device: Twister / harvesting

Adapted from quickstart.ipynb for the hardware computer. Edit Configuration, then Run All.
**Cells only print PowerShell commands:** paste selected commands into that computer's terminal.
Run All does not install packages, connect hardware, upload data or start training/inference.
Select the harvesting_hardware kernel after registering it. Any Python kernel can generate initial setup commands.
Skip installations and environment creation that are already complete.
Use matching embodiment, dataset/model IDs and Git commit on both computers.

In [ ]:
import json
from pathlib import Path

def find_repo(default: str) -> str:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/lerobot").is_dir():
            return str(candidate)
    return default

def ps_quote(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"

def flag(name: str, value) -> str:
    if isinstance(value, bool): text = str(value).lower()
    elif isinstance(value, dict): text = json.dumps(value)
    else: text = str(value)
    return f"--{name}={ps_quote(text)}"

def print_cmd(*parts: str) -> None:
    """Generate PowerShell commands; never execute them."""
    print(" `\n  ".join(part for part in parts if part))

def print_session(environment: str, repo_path: str) -> None:
    print(f"conda activate {environment}")
    print(f"Set-Location -LiteralPath {ps_quote(repo_path)}")
    print('$env:PYTHONUTF8 = "1"')

## Configuration
Windows/PowerShell commands are generated. COM numbers belong to this computer: discover and replace them.
For Linux, adapt shell quoting, activation and serial paths. Set the repo path if automatic discovery fails.
Use the same embodiment and dataset/model IDs as training_device.ipynb. Left is Twister; right is SO-100.
Keep device IDs, cameras and motion settings consistent across calibration, recording and deployment.
Restrict cumulative travel to what the mechanism and cabling permit.

In [ ]:
ENV_NAME = "harvesting_hardware"
REPO_PATH = find_repo(str(Path.cwd()))  # Edit for this hardware computer if necessary.
EMBODIMENT = "bimanual_harvesting"  # Or "twister".
assert EMBODIMENT in {"twister", "bimanual_harvesting"}
HF_USER = "YOUR_HF_USERNAME"
DATASET_NAME = f"{EMBODIMENT}_demo"
POLICY_NAME = f"{EMBODIMENT}_act"
DATASET_REPO_ID = f"{HF_USER}/{DATASET_NAME}"
POLICY_PATH = f"{HF_USER}/{POLICY_NAME}"  # Or copied pretrained_model folder on this computer.
ROBOT_ID = "harvesting_follower" if EMBODIMENT == "bimanual_harvesting" else "harvesting_follower_twister"
TELEOP_ID = "harvesting_leader" if EMBODIMENT == "bimanual_harvesting" else "harvesting_leader_twister"
TWISTER_FOLLOWER_PORT = "COM16"
TWISTER_LEADER_PORT = "COM14"
SO100_FOLLOWER_PORT = "COM15"
SO100_LEADER_PORT = "COM13"
TWISTER_SETTINGS = {"rotation_motor_id": 1, "gripper_motor_id": 2,
                    "rotation_control": "continuous", "rotation_min_deg": -720.0,
                    "rotation_max_deg": 720.0, "rotation_speed": 400}
CAMERAS = {"front": {"type": "opencv", "index_or_path": 0, "width": 640, "height": 480, "fps": 30}}
TASK_DESCRIPTION = "Align the object, assist with the gripper, then harvest." if EMBODIMENT == "bimanual_harvesting" else "Align the object, assist with the gripper, then twist further."
FPS = 30
NUM_EPISODES = 2  # Check recording first; increase for a useful demonstration dataset.
EPISODE_TIME_S = 30
RESET_TIME_S = 15
PUSH_DATASET_TO_HUB = True
DATASET_PRIVATE = False
DATASET_ROOT = f"data/{DATASET_NAME}"  # Same path in new recording and resume.
REPLAY_EPISODE = 0
INFERENCE_DEVICE = "cpu"  # Use cuda only if this computer's GPU checks succeed.
ROLLOUT_DURATION_S = 10
print(f"Repo: {REPO_PATH}\nEmbodiment: {EMBODIMENT}\nDataset: {DATASET_REPO_ID}\nPolicy: {POLICY_PATH}")
print("Edit the username, ports and camera index before running terminal commands.")

In [ ]:
def hardware_flags(role: str, *, cameras: bool = False) -> list[str]:
    assert role in {"robot", "teleop"}
    follower = role == "robot"
    kind = EMBODIMENT if follower else ("bimanual_harvesting_leader" if EMBODIMENT == "bimanual_harvesting" else "twister_leader")
    device_id = ROBOT_ID if follower else TELEOP_ID
    twister_port = TWISTER_FOLLOWER_PORT if follower else TWISTER_LEADER_PORT
    parts = [flag(f"{role}.type", kind), flag(f"{role}.id", device_id)]
    if EMBODIMENT == "bimanual_harvesting":
        parts += [flag(f"{role}.left_arm_config.port", twister_port),
                  flag(f"{role}.right_arm_config.port", SO100_FOLLOWER_PORT if follower else SO100_LEADER_PORT),
                  flag(f"{role}.right_arm_config.use_degrees", True)]
        prefix = f"{role}.left_arm_config"
    else:
        parts.append(flag(f"{role}.port", twister_port))
        prefix = role
    parts += [flag(f"{prefix}.{key}", value) for key, value in TWISTER_SETTINGS.items()]
    if follower and cameras and CAMERAS: parts.append(flag("robot.cameras", CAMERAS))
    return parts

def record_parts() -> list[str]:
    return ["lerobot-record", *hardware_flags("robot", cameras=True), *hardware_flags("teleop"),
            flag("dataset.repo_id", DATASET_REPO_ID), flag("dataset.root", DATASET_ROOT),
            flag("dataset.single_task", TASK_DESCRIPTION), flag("dataset.fps", FPS),
            flag("dataset.num_episodes", NUM_EPISODES), flag("dataset.episode_time_s", EPISODE_TIME_S),
            flag("dataset.reset_time_s", RESET_TIME_S), flag("dataset.no_stamp", True),
            flag("dataset.push_to_hub", PUSH_DATASET_TO_HUB), flag("dataset.private", DATASET_PRIVATE),
            flag("display_data", False), flag("play_sounds", False)]

## 1. Initialise once; activate each session
Create this environment locally on the hardware computer. It is independent of harvesting_train.
ACT inference uses base policy dependencies; additional policy types can require extras.
Register ipykernel to select this environment in VS Code. FFmpeg installation is optional for standard
PyAV wheels. No CUDA stack from the training computer is imposed on this computer.

In [ ]:
print(f"Set-Location -LiteralPath {ps_quote(REPO_PATH)}")
print(f"conda create -n {ENV_NAME} python=3.12 -y")
print_session(ENV_NAME, REPO_PATH)
print('python -m pip install -e ".[core_scripts,feetech]"')
print("python -m pip install ipykernel")
print(f'python -m ipykernel install --user --name {ENV_NAME} --display-name "{ENV_NAME} (Python 3.12)"')
print("# Optional: conda install -c conda-forge ffmpeg")
print("python -c " + ps_quote("import sys,lerobot; print(sys.executable); print(lerobot.__file__)"))
print("python examples/harvesting/verify_install.py")
print("hf auth login")
print("\n# Later terminal sessions:")
print_session(ENV_NAME, REPO_PATH)

## 2. Discover ports and camera
Repeat serial discovery for every controller, following the unplugging prompts. Update Configuration and
regenerate commands. Discover the camera index on this computer rather than copying another PC's index.

In [ ]:
print("lerobot-find-port")
print("lerobot-find-cameras opencv")

## 3. Motor IDs: first setup only
Skip devices whose IDs are already correct. Connect only the motor requested by the prompt; power off
before changing wiring. Twister defaults are rotation 1/gripper 2; SO-100 uses 1..6. IDs can repeat on
separate buses. Reconnect the complete motor chains before calibration.

In [ ]:
for role, port, device_id in [
    ("teleop", TWISTER_LEADER_PORT, TELEOP_ID + "_twister" if EMBODIMENT == "bimanual_harvesting" else TELEOP_ID),
    ("robot", TWISTER_FOLLOWER_PORT, ROBOT_ID + "_twister" if EMBODIMENT == "bimanual_harvesting" else ROBOT_ID),
]:
    print_cmd("lerobot-setup-motors", flag(f"{role}.type", "twister_leader" if role == "teleop" else "twister"),
              flag(f"{role}.port", port), flag(f"{role}.id", device_id),
              flag(f"{role}.rotation_motor_id", TWISTER_SETTINGS["rotation_motor_id"]),
              flag(f"{role}.gripper_motor_id", TWISTER_SETTINGS["gripper_motor_id"]))
    print()
if EMBODIMENT == "bimanual_harvesting":
    print_cmd("lerobot-setup-motors", flag("teleop.type", "so100_leader"),
              flag("teleop.port", SO100_LEADER_PORT), flag("teleop.id", TELEOP_ID + "_arm"))
    print()
    print_cmd("lerobot-setup-motors", flag("robot.type", "so100_follower"),
              flag("robot.port", SO100_FOLLOWER_PORT), flag("robot.id", ROBOT_ID + "_arm"))

## 4. Calibrate leaders, then followers
Follow the prompts. Twister calibration measures only gripper travel and must not cross the encoder seam;
rotation needs no home movement. Reuse valid calibration with ENTER. Keep the IDs below for later commands.
Calibration stays on the hardware device and is not needed for training.

In [ ]:
print_cmd("lerobot-calibrate", *hardware_flags("teleop"))
print()
print_cmd("lerobot-calibrate", *hardware_flags("robot"))

## 5. Teleoperate and check movement
Start leaders and followers in corresponding poses; keep them still during connection. Rotation establishes
session-relative zero independently on each Twister. Check slow movement and stopping before recording on
an object. Ctrl+C stops teleoperation.

In [ ]:
print_cmd("lerobot-teleoperate", *hardware_flags("robot"), *hardware_flags("teleop"),
          flag("fps", FPS), flag("display_data", False))

## 6. Record and upload
Use a fresh dataset name for a new recording. A local root is supplied explicitly and is reused for resume.
With upload enabled, login with write access and wait for upload to finish before starting training elsewhere.
Increase NUM_EPISODES after checking recording. Keep camera settings and starting poses consistent.
Turn count persists through episode resets in a connection; reconnection establishes a new reference.

In [ ]:
print_cmd(*record_parts())

### Optional: append/resume a recording
Use only with an existing compatible dataset at the same root/repository ID. This is an alternative to
starting a new recording, not a step to run immediately after every recording by default.

In [ ]:
print_cmd(*record_parts(), flag("resume", True))

## 7. Optional: replay an episode
This moves the followers using recorded actions. Match the physical starting arrangement and motion settings.
No leaders are needed. The dataset root matches the recording directory above.

In [ ]:
print_cmd("lerobot-replay", *hardware_flags("robot"), flag("dataset.repo_id", DATASET_REPO_ID),
          flag("dataset.root", DATASET_ROOT), flag("dataset.episode", REPLAY_EPISODE), flag("play_sounds", False))

## 8. Handoff to the training computer
Open training_device.ipynb there and use the dataset repository ID below. Train/upload there, then return here.
For local-only transfer, copy the complete dataset and set DATASET_LOCAL_ROOT in the training notebook.
Keep both computers on the same Git commit for the recording/training/deployment cycle.

In [ ]:
print(f"Training DATASET_REPO_ID = {DATASET_REPO_ID!r}")
print(f"Embodiment = {EMBODIMENT!r}")
print(f"Local directory for manual dataset transfer: {DATASET_ROOT}")
print(f"Expected deployed policy: {POLICY_PATH}")

## 9. Deploy the trained ACT policy
Wait for model upload. POLICY_PATH can be its Hub ID or a copied complete pretrained_model directory on this
computer. Match recording's camera names/dimensions, motion settings and task poses. Leaders are not required.
Begin with a short unloaded trial. CPU is the default; GPU training on another computer does not imply this
one has a GPU. Measure whether CPU inference sustains FPS and adjust it as needed. Select cuda only after
this computer's GPU check succeeds. The generated command disables automatic return to the initial pose.

In [ ]:
print("# Optional GPU check on this hardware computer:")
print("python -c " + ps_quote("import torch; print(torch.cuda.is_available())"))
print()
print_cmd("lerobot-rollout", *hardware_flags("robot", cameras=True),
          flag("policy.path", POLICY_PATH), flag("policy.device", INFERENCE_DEVICE),
          flag("strategy.type", "base"), flag("fps", FPS), flag("duration", ROLLOUT_DURATION_S),
          flag("task", TASK_DESCRIPTION), flag("return_to_initial_position", False),
          flag("display_data", False), flag("play_sounds", False))

## Local environments and shared source
Each computer creates a local Conda environment and logs in locally. Sharing Git source does not share
installed environments, caches or calibration. Keep code commits consistent and transfer datasets/models
deliberately. Avoid synchronizing environments or writing to the same output directory on a shared filesystem.
See ../../docs/harvesting/USE_CASE_CHANGELOG.md for the custom file inventory.